In [2]:
# ============================================================================
# IMPORT LIBRARIES
# ============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, KFold, train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import shapiro
import warnings
warnings.filterwarnings('ignore')

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

print("="*70)
print("MODEL 4 & BASELINE: MULTI-SPLIT ANALYSIS")
print("High-Quality Dataset (240 Matches)")
print("Table Format for Report")
print("="*70)

MODEL 4 & BASELINE: MULTI-SPLIT ANALYSIS
High-Quality Dataset (240 Matches)
Table Format for Report


In [3]:
# ============================================================================
# CONFIGURATION
# ============================================================================
TEST_SPLITS = [0.30, 0.20, 0.10]  # 70/30, 80/20, 90/10
RANDOM_SEED = 42
KFOLD_SPLITS = 5

print(f"\nConfiguration:")
print(f"  Test splits: {[f'{s*100:.0f}%' for s in TEST_SPLITS]}")
print(f"  Train splits: {[f'{(1-s)*100:.0f}%' for s in TEST_SPLITS]}")
print(f"  Random seed: {RANDOM_SEED}")


Configuration:
  Test splits: ['30%', '20%', '10%']
  Train splits: ['70%', '80%', '90%']
  Random seed: 42


In [4]:
# ============================================================================
# LOAD & PREPARE DATA (HIGH-QUALITY SUBSET)
# ============================================================================
print("\n[STEP 1] Loading Data")
print("-"*70)

matches_data = pd.read_csv('matchesk.csv')
external_data = pd.read_csv('external_factors.csv')
teams_data = pd.read_csv('teamsk.csv')
venues_data = pd.read_csv('venuesk.csv')
fifa_stats = pd.read_csv('FIFA_Site_Team_Stats.csv')

# Merge data
dataset = matches_data.merge(external_data, left_on='match_id', right_on='match_id', how='inner')
dataset['goal_diff'] = dataset['goals_home'] - dataset['goals_away']

# Add team names and ELO
team_id_to_name = dict(zip(teams_data['team_id'], teams_data['team_name']))
dataset['team_home_name'] = dataset['team_home'].map(team_id_to_name)
dataset['team_away_name'] = dataset['team_away'].map(team_id_to_name)

teams_elo_map = dict(zip(teams_data['team_id'], teams_data['elo_rating']))
dataset['home_elo_rating'] = dataset['team_home'].map(teams_elo_map)
dataset['away_elo_rating'] = dataset['team_away'].map(teams_elo_map)
avg_elo = teams_data['elo_rating'].mean()
dataset['home_elo_rating'].fillna(avg_elo, inplace=True)
dataset['away_elo_rating'].fillna(avg_elo, inplace=True)

# Add possession from FIFA stats (HIGH-QUALITY subset)
fifa_poss_map = dict(zip(fifa_stats['Team'], fifa_stats['Possession_Control(%)']))
dataset['home_Possession_fifa'] = dataset['team_home_name'].map(fifa_poss_map)
dataset['away_Possession_fifa'] = dataset['team_away_name'].map(fifa_poss_map)

# Filter to high-quality subset
dataset_hq = dataset[dataset['home_Possession_fifa'].notna()].copy()
dataset_hq['home_Possession'] = dataset_hq['home_Possession_fifa']
dataset_hq['away_Possession'] = dataset_hq['away_Possession_fifa']

# Add pitch quality
venues_map = dict(zip(venues_data['venue_id'].astype(str), venues_data['pitch_quality_score']))
dataset_hq['venue_id_str'] = dataset_hq['venue_id'].astype(str)
dataset_hq['pitch_quality'] = dataset_hq['venue_id_str'].map(venues_map)
avg_pitch = venues_data['pitch_quality_score'].mean()
dataset_hq['pitch_quality'].fillna(avg_pitch, inplace=True)

print(f"✓ Data loaded and filtered to {len(dataset_hq)} high-quality matches")


[STEP 1] Loading Data
----------------------------------------------------------------------
✓ Data loaded and filtered to 240 high-quality matches


In [5]:
# ============================================================================
# CREATE FEATURES
# ============================================================================
print("\n[STEP 2] Creating Features")
print("-"*70)

# Base features
X_base = pd.DataFrame()
X_base['elo_rating_diff'] = dataset_hq['home_elo_rating'] - dataset_hq['away_elo_rating']
X_base['xG_home'] = dataset_hq['xG_home']
X_base['xG_away'] = dataset_hq['xG_away']
X_base['home_Possession'] = dataset_hq['home_Possession']
X_base['travel_distance_km'] = dataset_hq['travel_distance_km']
X_base['betting_odds_home'] = dataset_hq['betting_odds_home']
X_base['pitch_quality'] = dataset_hq['pitch_quality']
X_base['referee_strictness'] = dataset_hq['referee_strictness']

# MODEL 4: Add polynomial & interaction terms
X_model4 = X_base.copy()
X_model4['elo_squared'] = X_base['elo_rating_diff'] ** 2
X_model4['xG_diff'] = X_base['xG_home'] - X_base['xG_away']
X_model4['xG_ratio'] = X_base['xG_home'] / (X_base['xG_away'] + 0.1)
X_model4['possession_x_elo'] = X_base['home_Possession'] * X_base['elo_rating_diff']
X_model4['possession_x_xG_diff'] = X_base['home_Possession'] * (X_base['xG_home'] - X_base['xG_away'])
X_model4['referee_x_travel'] = X_base['referee_strictness'] * X_base['travel_distance_km']

# BASELINE: Use only base features (no polynomial/interaction)
X_baseline = X_base.copy()

y = dataset_hq['goal_diff'].copy()

# Remove NaNs
clean_idx_m4 = X_model4.notna().all(axis=1) & y.notna()
X_model4 = X_model4[clean_idx_m4]
y_model4 = y[clean_idx_m4]

clean_idx_bl = X_baseline.notna().all(axis=1) & y.notna()
X_baseline = X_baseline[clean_idx_bl]
y_baseline = y[clean_idx_bl]

# Calculate normalization factor
mean_y_model4 = y_model4.mean()
mean_y_baseline = y_baseline.mean()

print(f"✓ Model 4: {X_model4.shape[1]} features, {len(X_model4)} samples")
print(f"✓ Baseline: {X_baseline.shape[1]} features, {len(X_baseline)} samples")


[STEP 2] Creating Features
----------------------------------------------------------------------
✓ Model 4: 14 features, 240 samples
✓ Baseline: 8 features, 240 samples


In [6]:
# ============================================================================
# MULTI-SPLIT ANALYSIS - MODEL 4 & BASELINE
# ============================================================================
print("\n" + "="*70)
print("[EVALUATING MODELS ACROSS SPLITS]")
print("="*70)

# Prepare data
X_model4_scaled = StandardScaler().fit_transform(X_model4)
X_baseline_scaled = StandardScaler().fit_transform(X_baseline)

# Store results
table_data = []

for test_size in TEST_SPLITS:
    train_size = 1 - test_size
    print(f"\n--- Split: {train_size*100:.0f}% Training / {test_size*100:.0f}% Test ---")
    
    # ===== MODEL 4 =====
    X_train_m4, X_test_m4, y_train_m4, y_test_m4 = train_test_split(
        X_model4_scaled, y_model4, test_size=test_size, random_state=RANDOM_SEED
    )
    
    model_m4 = LinearRegression()
    model_m4.fit(X_train_m4, y_train_m4)
    y_pred_m4 = model_m4.predict(X_test_m4)
    
    # Calculate metrics for Model 4
    mae_m4 = mean_absolute_error(y_test_m4, y_pred_m4)
    mse_m4 = mean_squared_error(y_test_m4, y_pred_m4)
    rmse_m4 = np.sqrt(mse_m4)
    rmse_norm_m4 = rmse_m4 / mean_y_model4 if mean_y_model4 != 0 else 0
    r2_m4 = r2_score(y_test_m4, y_pred_m4)
    intercept_m4 = model_m4.intercept_
    
    print(f"  Model 4: R²={r2_m4:.4f}, RMSE={rmse_m4:.4f}, MAE={mae_m4:.4f}")
    
    # ===== BASELINE =====
    X_train_bl, X_test_bl, y_train_bl, y_test_bl = train_test_split(
        X_baseline_scaled, y_baseline, test_size=test_size, random_state=RANDOM_SEED
    )
    
    model_bl = LinearRegression()
    model_bl.fit(X_train_bl, y_train_bl)
    y_pred_bl = model_bl.predict(X_test_bl)
    
    # Calculate metrics for Baseline
    mae_bl = mean_absolute_error(y_test_bl, y_pred_bl)
    mse_bl = mean_squared_error(y_test_bl, y_pred_bl)
    rmse_bl = np.sqrt(mse_bl)
    rmse_norm_bl = rmse_bl / mean_y_baseline if mean_y_baseline != 0 else 0
    r2_bl = r2_score(y_test_bl, y_pred_bl)
    intercept_bl = model_bl.intercept_
    
    print(f"  Baseline: R²={r2_bl:.4f}, RMSE={rmse_bl:.4f}, MAE={mae_bl:.4f}")
    
    # Store for table
    table_data.append({
        'Split': f"{train_size*100:.0f}% Training / {test_size*100:.0f}% Test",
        'Model_4_Intercept': intercept_m4,
        'Model_4_MAE': mae_m4,
        'Model_4_MSE': mse_m4,
        'Model_4_RMSE': rmse_m4,
        'Model_4_RMSE_Norm': rmse_norm_m4,
        'Model_4_R2': r2_m4,
        'Baseline_Intercept': intercept_bl,
        'Baseline_MAE': mae_bl,
        'Baseline_MSE': mse_bl,
        'Baseline_RMSE': rmse_bl,
        'Baseline_RMSE_Norm': rmse_norm_bl,
        'Baseline_R2': r2_bl
    })

print("\n✓ Analysis complete")


[EVALUATING MODELS ACROSS SPLITS]

--- Split: 70% Training / 30% Test ---
  Model 4: R²=0.8717, RMSE=0.4994, MAE=0.4305
  Baseline: R²=0.3984, RMSE=1.0814, MAE=0.8028

--- Split: 80% Training / 20% Test ---
  Model 4: R²=0.8539, RMSE=0.4993, MAE=0.4373
  Baseline: R²=0.3584, RMSE=1.0464, MAE=0.7928

--- Split: 90% Training / 10% Test ---
  Model 4: R²=0.8014, RMSE=0.4927, MAE=0.4348
  Baseline: R²=0.2993, RMSE=0.9254, MAE=0.6873

✓ Analysis complete


In [7]:
# ============================================================================
# CREATE FORMATTED TABLE (Matches your report format)
# ============================================================================
print("\n" + "="*70)
print("[TABLE 3: RESULTS OF TEST/TRAIN MODELS]")
print("="*70)

# Create table matching your format
formatted_table = pd.DataFrame([
    {
        'Metric': 'Test Size',
        '70% Training / 30% Test': table_data[0]['Split'].split('/')[1].strip(),
        '80% Training / 20% Test': table_data[1]['Split'].split('/')[1].strip(),
        '90% Training / 10% Test': table_data[2]['Split'].split('/')[1].strip()
    },
    {
        'Metric': 'Intercept (Model 4)',
        '70% Training / 30% Test': f"{table_data[0]['Model_4_Intercept']:.4f}",
        '80% Training / 20% Test': f"{table_data[1]['Model_4_Intercept']:.4f}",
        '90% Training / 10% Test': f"{table_data[2]['Model_4_Intercept']:.4f}"
    },
    {
        'Metric': 'MAE (Model 4)',
        '70% Training / 30% Test': f"{table_data[0]['Model_4_MAE']:.4f}",
        '80% Training / 20% Test': f"{table_data[1]['Model_4_MAE']:.4f}",
        '90% Training / 10% Test': f"{table_data[2]['Model_4_MAE']:.4f}"
    },
    {
        'Metric': 'MSE (Model 4)',
        '70% Training / 30% Test': f"{table_data[0]['Model_4_MSE']:.4f}",
        '80% Training / 20% Test': f"{table_data[1]['Model_4_MSE']:.4f}",
        '90% Training / 10% Test': f"{table_data[2]['Model_4_MSE']:.4f}"
    },
    {
        'Metric': 'RMSE (Model 4)',
        '70% Training / 30% Test': f"{table_data[0]['Model_4_RMSE']:.4f}",
        '80% Training / 20% Test': f"{table_data[1]['Model_4_RMSE']:.4f}",
        '90% Training / 10% Test': f"{table_data[2]['Model_4_RMSE']:.4f}"
    },
    {
        'Metric': 'RMSE_Norm (Model 4)',
        '70% Training / 30% Test': f"{table_data[0]['Model_4_RMSE_Norm']:.4f}",
        '80% Training / 20% Test': f"{table_data[1]['Model_4_RMSE_Norm']:.4f}",
        '90% Training / 10% Test': f"{table_data[2]['Model_4_RMSE_Norm']:.4f}"
    },
    {
        'Metric': 'R² (Model 4)',
        '70% Training / 30% Test': f"{table_data[0]['Model_4_R2']:.4f}",
        '80% Training / 20% Test': f"{table_data[1]['Model_4_R2']:.4f}",
        '90% Training / 10% Test': f"{table_data[2]['Model_4_R2']:.4f}"
    },
    {'Metric': '', '70% Training / 30% Test': '', '80% Training / 20% Test': '', '90% Training / 10% Test': ''},
    {
        'Metric': 'Intercept (Baseline)',
        '70% Training / 30% Test': f"{table_data[0]['Baseline_Intercept']:.4f}",
        '80% Training / 20% Test': f"{table_data[1]['Baseline_Intercept']:.4f}",
        '90% Training / 10% Test': f"{table_data[2]['Baseline_Intercept']:.4f}"
    },
    {
        'Metric': 'MAE (Baseline)',
        '70% Training / 30% Test': f"{table_data[0]['Baseline_MAE']:.4f}",
        '80% Training / 20% Test': f"{table_data[1]['Baseline_MAE']:.4f}",
        '90% Training / 10% Test': f"{table_data[2]['Baseline_MAE']:.4f}"
    },
    {
        'Metric': 'MSE (Baseline)',
        '70% Training / 30% Test': f"{table_data[0]['Baseline_MSE']:.4f}",
        '80% Training / 20% Test': f"{table_data[1]['Baseline_MSE']:.4f}",
        '90% Training / 10% Test': f"{table_data[2]['Baseline_MSE']:.4f}"
    },
    {
        'Metric': 'RMSE (Baseline)',
        '70% Training / 30% Test': f"{table_data[0]['Baseline_RMSE']:.4f}",
        '80% Training / 20% Test': f"{table_data[1]['Baseline_RMSE']:.4f}",
        '90% Training / 10% Test': f"{table_data[2]['Baseline_RMSE']:.4f}"
    },
    {
        'Metric': 'RMSE_Norm (Baseline)',
        '70% Training / 30% Test': f"{table_data[0]['Baseline_RMSE_Norm']:.4f}",
        '80% Training / 20% Test': f"{table_data[1]['Baseline_RMSE_Norm']:.4f}",
        '90% Training / 10% Test': f"{table_data[2]['Baseline_RMSE_Norm']:.4f}"
    },
    {
        'Metric': 'R² (Baseline)',
        '70% Training / 30% Test': f"{table_data[0]['Baseline_R2']:.4f}",
        '80% Training / 20% Test': f"{table_data[1]['Baseline_R2']:.4f}",
        '90% Training / 10% Test': f"{table_data[2]['Baseline_R2']:.4f}"
    }
])

print("\n" + formatted_table.to_string(index=False))

# Save formatted table
formatted_table.to_csv('Table_3_Results_MultiSplit.csv', index=False)
print("\n✓ Saved: Table_3_Results_MultiSplit.csv (Ready for report)")


[TABLE 3: RESULTS OF TEST/TRAIN MODELS]

              Metric 70% Training / 30% Test 80% Training / 20% Test 90% Training / 10% Test
           Test Size                30% Test                20% Test                10% Test
 Intercept (Model 4)                 -0.1427                 -0.1331                 -0.1338
       MAE (Model 4)                  0.4305                  0.4373                  0.4348
       MSE (Model 4)                  0.2494                  0.2493                  0.2427
      RMSE (Model 4)                  0.4994                  0.4993                  0.4927
 RMSE_Norm (Model 4)                 -3.7452                 -3.7445                 -3.6949
        R² (Model 4)                  0.8717                  0.8539                  0.8014
                                                                                            
Intercept (Baseline)                 -0.1180                 -0.1142                 -0.1557
      MAE (Baseline)        

In [8]:
# ============================================================================
# SUMMARY & INTERPRETATION
# ============================================================================
print("\n" + "="*70)
print("[SUMMARY & INTERPRETATION]")
print("="*70)

print(f"""
MODEL 4 vs BASELINE COMPARISON:

1. BEST PERFORMANCE (Model 4):
   Test Size 80/20:
   - R²: {table_data[1]['Model_4_R2']:.4f} (Model 4) vs {table_data[1]['Baseline_R2']:.4f} (Baseline)
   - RMSE: {table_data[1]['Model_4_RMSE']:.4f} (Model 4) vs {table_data[1]['Baseline_RMSE']:.4f} (Baseline)
   - Improvement: {((table_data[1]['Model_4_R2'] - table_data[1]['Baseline_R2']) / table_data[1]['Baseline_R2'] * 100):.1f}% R² improvement

2. ROBUSTNESS ACROSS SPLITS:
   - Model 4 R² Mean: {np.mean([d['Model_4_R2'] for d in table_data]):.4f}
   - Baseline R² Mean: {np.mean([d['Baseline_R2'] for d in table_data]):.4f}
   - Model 4 is consistently better across all splits

3. NORMALIZED RMSE (RMSE / mean(y)):
   - Shows prediction error as proportion of mean goal difference
   - Model 4 RMSE_Norm: {np.mean([d['Model_4_RMSE_Norm'] for d in table_data]):.4f} (average)
   - Baseline RMSE_Norm: {np.mean([d['Baseline_RMSE_Norm'] for d in table_data]):.4f} (average)

4. RECOMMENDATION:
   ✓ Use 80/20 split for primary results
   ✓ Model 4 significantly outperforms Baseline
   ✓ Performance stable across different test sizes
   ✓ High R² validates feature engineering approach
""")

print("="*70)
print("ANALYSIS COMPLETE ✓")
print("="*70)


[SUMMARY & INTERPRETATION]

MODEL 4 vs BASELINE COMPARISON:

1. BEST PERFORMANCE (Model 4):
   Test Size 80/20:
   - R²: 0.8539 (Model 4) vs 0.3584 (Baseline)
   - RMSE: 0.4993 (Model 4) vs 1.0464 (Baseline)
   - Improvement: 138.2% R² improvement

2. ROBUSTNESS ACROSS SPLITS:
   - Model 4 R² Mean: 0.8424
   - Baseline R² Mean: 0.3520
   - Model 4 is consistently better across all splits

3. NORMALIZED RMSE (RMSE / mean(y)):
   - Shows prediction error as proportion of mean goal difference
   - Model 4 RMSE_Norm: -3.7282 (average)
   - Baseline RMSE_Norm: -7.6329 (average)

4. RECOMMENDATION:
   ✓ Use 80/20 split for primary results
   ✓ Model 4 significantly outperforms Baseline
   ✓ Performance stable across different test sizes
   ✓ High R² validates feature engineering approach

ANALYSIS COMPLETE ✓
